# TALLER DE PROCESAMIENTO DE DATOS - MACHINE LEARNING
**Universidad Libre**

Base de datos: `VENTAS_NL.xlsx`

**Integrantes:**
- Jarol Steven Gutierrez Gordillo (LIDER)
- Daniel Mauricio Agreda Aguilar

**Fecha:** 15 de agosto de 2026

Script ejecutable de principio a fin. Reproduce las 4 secciones del taller:
1. Limpieza de datos
2. Estandarizacion
3. Ingenieria de caracteristicas
4. Analisis y propuesta de modelo de prediccion


In [ ]:
# -*- coding: utf-8 -*-
import pandas as pd
import numpy as np
from scipy.stats import skew
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

RUTA_ARCHIVO = "VENTAS_NL.xlsx"  # Ajustar ruta segun donde se ejecute


## SECCION 1: LIMPIEZA DE DATOS

### 1.1 Carga de datos

In [ ]:
df = pd.read_excel(RUTA_ARCHIVO)
n_original = len(df)
print(f"Registros originales: {n_original}")


### 1.2 Duplicados
Se identifican filas identicas en todas sus columnas y se eliminan.
Un duplicado infla artificialmente estadisticas como promedios y sumas.

In [ ]:
n_dup = df.duplicated().sum()
df = df.drop_duplicates().reset_index(drop=True)
print(f"Duplicados eliminados: {n_dup}")
print(f"Registros tras eliminar duplicados: {len(df)}")


### 1.3 Validacion y correccion de formatos

In [ ]:
# id: debe ser entero y unico
df['id'] = df['id'].astype(int)
assert df['id'].duplicated().sum() == 0, "Aun hay ids duplicados"

# EDAD: entero, rango logico de edad adulta
df['EDAD'] = df['EDAD'].astype(int)
assert df['EDAD'].between(0, 100).all(), "Hay edades fuera de rango"

# GENERO: texto categorico. Se limpia espacios y se unifica mayusculas
# para evitar que 'M', 'm', ' M ' se traten como categorias distintas.
df['GENERO'] = df['GENERO'].astype(str).str.strip().str.upper()

# SIZE: categoria codificada como entero (1 a 5)
df['SIZE'] = df['SIZE'].astype(int)

# Variables numericas continuas: forzar tipo float
for col in ['YEARINCOME', 'ventas', 'costo venta', 'DESCUENTOS']:
    df[col] = pd.to_numeric(df[col], errors='coerce')


### 1.4 Datos faltantes + Skewness + Imputacion
Para cada columna con datos faltantes se calcula el coeficiente de
asimetria (skewness) de los valores NO faltantes. Regla de decision:
- `|skewness| <= 0.5` -> distribucion aprox. simetrica -> se imputa con la **MEDIA**
- `|skewness| > 0.5` -> distribucion asimetrica -> se imputa con la **MEDIANA**
  (la mediana es robusta a valores extremos que "arrastran" la media)

In [ ]:
cols_faltantes = ['YEARINCOME', 'ventas', 'costo venta', 'DESCUENTOS']
reporte_imputacion = []

for col in cols_faltantes:
    n_missing = df[col].isna().sum()
    sk = skew(df[col].dropna())
    if abs(sk) <= 0.5:
        metodo = 'media'
        valor_imputacion = df[col].mean()
    else:
        metodo = 'mediana'
        valor_imputacion = df[col].median()
    df[col] = df[col].fillna(valor_imputacion)
    reporte_imputacion.append({
        'columna': col,
        'faltantes': int(n_missing),
        'skewness': round(sk, 3),
        'metodo': metodo,
        'valor_usado': round(valor_imputacion, 4)
    })

reporte_imputacion_df = pd.DataFrame(reporte_imputacion)
print("\nReporte de imputacion de datos faltantes:")
print(reporte_imputacion_df.to_string(index=False))

# Verificacion: ya no deben quedar valores nulos
assert df[cols_faltantes].isna().sum().sum() == 0, "Aun quedan valores nulos"


## SECCION 2: ESTANDARIZACION

- **StandardScaler**: centra en media 0 y desviacion estandar 1. Se aplica a `YEARINCOME` y `ventas`
  porque estan en escalas muy distintas entre si y respecto a otras variables (`EDAD`, `SIZE`), lo cual
  distorsiona a los modelos que son sensibles a la magnitud de los numeros.
- **MinMaxScaler**: comprime los valores a un rango fijo [0, 1]. Se aplica a `costo venta` para tener
  una version normalizada uniforme, util para comparar proporcionalmente sin importar la unidad original.

In [ ]:
scaler_std = StandardScaler()
df[['YEARINCOME_std', 'ventas_std']] = scaler_std.fit_transform(df[['YEARINCOME', 'ventas']])

scaler_mm = MinMaxScaler()
df['costo_venta_norm'] = scaler_mm.fit_transform(df[['costo venta']])

print("\nEjemplo de columnas estandarizadas:")
print(df[['YEARINCOME', 'YEARINCOME_std', 'ventas', 'ventas_std',
          'costo venta', 'costo_venta_norm']].head())


## SECCION 3: INGENIERIA DE CARACTERISTICAS

**Variable 1: `ventas_netas`**
Representa lo que el cliente realmente termino pagando despues de aplicar su descuento. Es mas fiel
al ingreso real de la empresa que la columna `ventas` bruta, y sirve como variable objetivo alternativa
o como insumo para calcular margenes reales.

**Variable 2 (no obvia): `indice_gasto_relativo`**
`indice_gasto_relativo = ventas / YEARINCOME`

No mide cuanto gasto el cliente en pesos absolutos, sino que porcentaje de su ingreso ANUAL representa
esa unica compra. Un cliente con ingreso bajo que gasta un 15% de su ingreso anual en una compra muestra
un compromiso/lealtad mucho mayor que un cliente rico que gasta la misma cantidad de pesos pero eso es
solo el 0.3% de lo que gana. Sirve para segmentar clientes por "intensidad de consumo" en vez de solo
por poder adquisitivo bruto, algo util para campanas de fidelizacion.

In [ ]:
df['ventas_netas'] = df['ventas'] * (1 - df['DESCUENTOS'])
df['indice_gasto_relativo'] = df['ventas'] / df['YEARINCOME']

print("\nNuevas variables creadas (muestra):")
print(df[['ventas', 'DESCUENTOS', 'ventas_netas', 'YEARINCOME',
          'indice_gasto_relativo']].head())


## SECCION 4: ANALISIS Y PROPUESTA DE MODELO DE PREDICCION

### 4.1 Matriz de correlacion

In [ ]:
cols_corr = ['EDAD', 'SIZE', 'YEARINCOME', 'ventas', 'costo venta',
             'DESCUENTOS', 'ventas_netas', 'indice_gasto_relativo']
corr_matrix = df[cols_corr].corr().round(3)
print("\nMatriz de correlacion:")
print(corr_matrix)


### 4.2 Variable objetivo y predictoras

Objetivo: `ventas`
Predictoras: `EDAD`, `GENERO`, `SIZE`, `YEARINCOME`, `DESCUENTOS`

Se **EXCLUYEN** explicitamente de las predictoras:
- `costo venta` -> correlacion ~0.94 con ventas (fuga de datos)
- `ventas_netas` -> se calcula directamente a partir de `ventas`
- `indice_gasto_relativo` -> se calcula directamente a partir de `ventas`

Usar cualquiera de estas como predictora seria "filtrar" la respuesta al modelo (data leakage), dando
una precision artificialmente alta que no se sostendria con datos nuevos reales.

In [ ]:
model_df = df.copy()
model_df['GENERO_enc'] = model_df['GENERO'].map({'M': 0, 'H': 1})

X = model_df[['EDAD', 'GENERO_enc', 'SIZE', 'YEARINCOME', 'DESCUENTOS']]
y = model_df['ventas']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


### 4.3 Prueba exploratoria: dos modelos de ensamble

Se prueban DOS algoritmos de arboles en ensamble (mas robustos ante relaciones no lineales y variables
asimetricas que una regresion lineal):
- Random Forest Regressor
- Gradient Boosting Regressor

y se comparan sus metricas para tener mayor exactitud/confianza en la conclusion sobre que tan
predecible es `ventas` con estas variables.

In [ ]:
modelos = {
    'Random Forest': RandomForestRegressor(
        n_estimators=200, max_depth=8, random_state=42, n_jobs=-1
    ),
    'Gradient Boosting': GradientBoostingRegressor(
        n_estimators=200, max_depth=3, learning_rate=0.1, random_state=42
    )
}

resultados = []
importancias = {}

for nombre, modelo in modelos.items():
    modelo.fit(X_train, y_train)
    pred = modelo.predict(X_test)
    r2 = r2_score(y_test, pred)
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    mae = mean_absolute_error(y_test, pred)
    resultados.append({'modelo': nombre, 'R2': round(r2, 4),
                        'RMSE': round(rmse, 2), 'MAE': round(mae, 2)})
    importancias[nombre] = dict(zip(X.columns, modelo.feature_importances_.round(4)))

resultados_df = pd.DataFrame(resultados)
print("\nResultados de los modelos (conjunto de prueba):")
print(resultados_df.to_string(index=False))
print("\nImportancia de variables por modelo:")
print(pd.DataFrame(importancias))


### 4.4 Conclusion de la prueba exploratoria

Ambos modelos obtienen un R2 cercano a 0, lo que indica que `EDAD`, `GENERO`, `SIZE`, `YEARINCOME` y
`DESCUENTOS` por si solos NO logran explicar la variabilidad del valor de venta. Esto sugiere que, para
un modelo de produccion, se necesitarian variables adicionales no presentes en esta base (categoria de
producto, canal de venta, temporada, etc.).

### 4.5 Exportar resultados a Excel

In [ ]:
with pd.ExcelWriter('VENTAS_NL_procesado.xlsx', engine='openpyxl') as writer:
    reporte_imputacion_df.to_excel(writer, sheet_name='Imputacion', index=False)
    df.to_excel(writer, sheet_name='Datos_Procesados', index=False)
    corr_matrix.to_excel(writer, sheet_name='Correlacion')
    resultados_df.to_excel(writer, sheet_name='Resultados_Modelos', index=False)

print("\nProceso completado. Archivo 'VENTAS_NL_procesado.xlsx' generado.")
